# J-lens Antidoom baseline (200 prompts, max_new_tokens=4000)

**Model:** `Qwen/Qwen3.5-4B` (4-bit NF4)  
**Dataset:** `LiquidAI/antidoom-mix-v1.0` — 7 reasoning sources only, seed=42  
**No training** — inference + Liquid loop detection + J-lens Tier-1/2 cache

Runtime → Change runtime type → **GPU (T4 or better)**.

This notebook mirrors `scripts/run_antidoom_200.py` with `JLENS_MAX_NEW_TOKENS=4000`.

## 0) Setup — clone / upload repo

Pick **one** path:
- **A:** clone from GitHub (if the repo is public / you have a token)
- **B:** upload a zip of the project to Drive and unzip

In [1]:
import os
from pathlib import Path

# === EDIT THESE ===
REPO_URL = ""  # e.g. https://github.com/<you>/j-lens.git  (leave empty to use Drive zip)
DRIVE_ZIP = ""  # e.g. /content/drive/MyDrive/j-lens.zip
PROJECT = Path("/content/j-lens")

USE_DRIVE = True  # mount Google Drive for results persistence

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if REPO_URL:
    !git clone {REPO_URL} {PROJECT}
elif DRIVE_ZIP:
    !unzip -q -o "{DRIVE_ZIP}" -d /content
    # adjust if zip contains a top-level folder name other than j-lens
    if not PROJECT.is_dir():
        cands = list(Path("/content").glob("*/jspace"))
        assert cands, "Could not find jspace/ after unzip — set PROJECT manually"
        PROJECT = cands[0].parent
else:
    raise RuntimeError("Set REPO_URL or DRIVE_ZIP so Colab can see the j-lens code.")

os.chdir(PROJECT)
print("cwd:", Path.cwd())
print("has jspace:", (PROJECT / "jspace").is_dir())

Mounted at /content/drive


RuntimeError: Set REPO_URL or DRIVE_ZIP so Colab can see the j-lens code.

In [ ]:
# Install deps (Colab already has torch+CUDA; keep bitsandbytes for 4-bit)
%pip install -q -U transformers accelerate bitsandbytes datasets huggingface_hub \
    pandas matplotlib seaborn scipy statsmodels tqdm pyyaml safetensors sentencepiece

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if torch.cuda.is_available() else None)

## 1) Config — 4000 tokens, same 200-prompt sample

In [ ]:
import os
from pathlib import Path

os.environ["JLENS_BASELINE_PROMPTS"] = "200"
os.environ["JLENS_SAMPLE_SEED"] = "42"
os.environ["JLENS_MAX_NEW_TOKENS"] = "4000"   # Liquid default.yaml
os.environ["JLENS_TEMPERATURE"] = "0.01"
os.environ["JLENS_EXP2_ANALYZE_ONLY"] = "1"
os.environ["PYTHONPATH"] = str(Path.cwd())

# Persist results to Drive if mounted
DRIVE_RESULTS = Path("/content/drive/MyDrive/j-lens-results-4000")
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print("max_new_tokens=", os.environ["JLENS_MAX_NEW_TOKENS"])
print("drive_results=", DRIVE_RESULTS)

## 2) Workspace band (skip if already present)

Needs the pre-fitted lens from neuronpedia (~387MB).

In [ ]:
from pathlib import Path
band = Path("results/workspace_band_qwen3.5-4b.json")
if band.is_file():
    print("workspace band already exists:", band)
else:
    !python scripts/01_workspace_band.py

## 3) Baseline pass (generation + loop detect + Tier-1/2)

This is the long cell. Resume-safe via `results/checkpoints/baseline_pass.json`.

On a Colab T4, expect roughly **~1–3 min/prompt** at 4000 tokens (varies). Full 200 ≈ a few hours.

In [ ]:
!python scripts/02_baseline_pass.py

## 4) Exp1 → Exp2 analyze → Exp3 (looping prompts only)

In [ ]:
!python scripts/03_exp1_static_geometry.py
!python scripts/04_exp2_dynamic.py
!python scripts/05_exp3_causal.py
!python -c "from scripts.run_antidoom_200 import write_report; print(write_report())"

## 5) Quick summary + sync to Drive

In [ ]:
import json, shutil
from pathlib import Path

summary = Path("results/baseline_pass_summary.json")
if summary.is_file():
    s = json.loads(summary.read_text())
    print("prompts:", s.get("n_prompts"), "loops:", s.get("n_loop"), "rate:", s.get("loop_rate"))
    print("max_new_tokens:", s.get("max_new_tokens"))
else:
    print("baseline summary missing — pass not finished")

# Copy key artifacts to Drive
for rel in [
    "results/baseline_pass_summary.json",
    "results/prompt_sample_ids.json",
    "results/trigger_tokens_qwen3.5-4b.csv",
    "results/trigger_gen_log.jsonl",
    "results/RUN_REPORT_antidoom_mix_200.md",
    "results/workspace_band_qwen3.5-4b.json",
]:
    src = Path(rel)
    if src.is_file():
        dst = DRIVE_RESULTS / src.name
        shutil.copy2(src, dst)
        print("saved", dst)

# Zip full results folder for download
zip_path = DRIVE_RESULTS / "results_full"
shutil.make_archive(str(zip_path), "zip", "results")
print("zip:", str(zip_path) + ".zip")

## Optional: progress monitor cell

Re-run anytime while baseline is running in another cell / background.

In [ ]:
import json
from pathlib import Path

ckpt = Path("results/checkpoints/baseline_pass.json")
log = Path("results/trigger_gen_log.jsonl")
if ckpt.is_file():
    n = len(json.loads(ckpt.read_text())["completed_prompt_ids"])
    print(f"checkpoint: {n}/200")
else:
    print("no checkpoint yet")
if log.is_file():
    loops = sum(1 for line in log.open(encoding="utf-8") if '"is_loop": true' in line)
    total = sum(1 for _ in log.open(encoding="utf-8"))
    print(f"log: {total} rows, loops={loops}, rate={loops/max(total,1):.1%}")